# Inference with a Distilled Deployment

Hit a local llama-server deployment launched from the Deployments UI.
The deployment exposes an OpenAI-compatible endpoint through the proxy at
`/v1/deployments/{deployment_id}/v1/chat/completions`.

Requests routed through the proxy are logged to `llm_traces` and show up in the Traces view.

In [ ]:
import requests

API_BASE = "http://localhost:8000"

# Auto-pick the most recently updated in_service deployment so this notebook
# survives redeploys. Override manually if you want a specific one.
deployments = requests.get(f"{API_BASE}/v1/deployments", timeout=10).json()["deployments"]
live = [d for d in deployments if d.get("status") == "in_service"]
if not live:
    raise RuntimeError("No in_service deployments — deploy one from the UI first")
live.sort(key=lambda d: d.get("updated_at", ""), reverse=True)
DEPLOYMENT_ID = live[0]["deployment_id"]
MODEL_ID = live[0]["model_id"]
print(f"using {DEPLOYMENT_ID} ({MODEL_ID})")

## 1. List all deployments

In [ ]:
resp = requests.get(f"{API_BASE}/v1/deployments", timeout=10).json()
for d in resp["deployments"]:
    print(f"{d['deployment_id']:<22} {d['status']:<12} {d['model_id']:<28} {d.get('endpoint_url','')}")

## 2. Chat completion via the deployment proxy

In [ ]:
payload = {
    "model": MODEL_ID,
    "messages": [
        {"role": "system", "content": "You are a concise assistant."},
        {"role": "user", "content": "In one sentence: what is knowledge distillation?"},
    ],
    "max_tokens": 128,
    "temperature": 0.7,
}

r = requests.post(
    f"{API_BASE}/v1/deployments/{DEPLOYMENT_ID}/v1/chat/completions",
    json=payload,
    timeout=60,
)
r.raise_for_status()
data = r.json()

print(data["choices"][0]["message"]["content"])
print("\n---")
print("usage:", data["usage"])
timings = data.get("timings", {})
if timings:
    print(f"prompt: {timings.get('prompt_per_second', 0):.1f} tok/s | predict: {timings.get('predicted_per_second', 0):.1f} tok/s")

## 3. Use the OpenAI SDK against the same proxy

In [ ]:
from openai import OpenAI

client = OpenAI(
    base_url=f"{API_BASE}/v1/deployments/{DEPLOYMENT_ID}/v1",
    api_key="not-needed",
)

completion = client.chat.completions.create(
    model=MODEL_ID,
    messages=[{"role": "user", "content": "Write a haiku about small models."}],
    max_tokens=64,
)
print(completion.choices[0].message.content)

## 4. Streaming

In [ ]:
stream = client.chat.completions.create(
    model=MODEL_ID,
    messages=[{"role": "user", "content": "Count from 1 to 5, one per line."}],
    max_tokens=64,
    stream=True,
)
for chunk in stream:
    delta = chunk.choices[0].delta.content or ""
    print(delta, end="", flush=True)
print()

## 5. Per-deployment metrics

In [ ]:
m = requests.get(f"{API_BASE}/v1/deployments/{DEPLOYMENT_ID}/metrics", timeout=10).json()
print(m)

## 6. Confirm the calls landed in the Traces view

In [ ]:
traces = requests.get(
    f"{API_BASE}/v1/traces",
    params={"model": MODEL_ID, "limit": 5},
    timeout=10,
).json()
rows = traces if isinstance(traces, list) else traces.get("traces", [])
for t in rows:
    preview = (t.get("output_text") or "")[:60]
    print(f"{t.get('request_id','')[:8]}  {t.get('request_type'):<6} stream={t.get('is_stream',0)}  "
          f"{t.get('latency_ms',0):>6.0f}ms  -> {preview!r}")